# Evaluation: main table, turbulence evaluation

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys


sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
device = "cuda"

In [3]:
from typing import Sequence, Optional, Callable, Dict

import torch
from scipy.stats import pearsonr
from collections import defaultdict
import matplotlib
import matplotlib.pyplot as plt
import warnings
import pickle
from tqdm import tqdm
import numpy as np
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore", message="__array_wrap__ must accept context*")

from neugk.dataset import CycloneAEDataset
from neugk.integrals import FluxIntegral

from neugk.pinc.autoencoders.ae_utils import load_autoencoder
from neugk.pinc.neural_fields.trad import (
    zfp_recon,
    wavelet_recon,
    pca_recon,
    jpeg2000_recon,
)
from neugk.pinc.neural_fields.data import CycloneNFDataset
from neugk.pinc.neural_fields.nf_utils import (
    sample_field,
    load_nf,
    compress_weights,
    endpoint_error,
)
from neugk.pinc.neural_fields.gk_losses import diagnostics

In [ ]:
NAME_TIMESTEPS = [
    100,
    105,
    110,
    115,
    120,
    125,
    130,
    135,
    140,
    145,
    150,
    155,
    160,
    165,
    170,
    175,
    180,
    185,
    190,
    195,
    200,
]
TIMESTEPS = range(len(NAME_TIMESTEPS))
TRAJECTORIES = [
    "iteration_13",
    "iteration_115",
    "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    "iteration_200",
    "iteration_210",
    "iteration_212",
    "iteration_213",
    "iteration_214",
    "iteration_215",
    "iteration_216",
    "iteration_218",
    "iteration_220",
    "iteration_224",
    "iteration_228",
    "iteration_230",
    "iteration_232",
]

BASE_PATH = "<path>"
# snapshot_download(repo_id="gerkone/pink_gkw", local_dir=BASE_PATH, local_dir_use_symlinks=False)

## Load all models

### Neural fields

In [6]:
CKP_DIR = f"{BASE_PATH}/nf_ckps_hf"

nfs = {}
for traj in TRAJECTORIES:
    nfs[traj] = {}
    for t, tn in zip(TIMESTEPS, NAME_TIMESTEPS):
        ckp_name = f"mlp_{traj}_t{tn}_x1167.pt"
        nfs[traj][t] = load_nf(f"{CKP_DIR}/{ckp_name}", device)

int_nfs = {}
for traj in TRAJECTORIES:
    int_nfs[traj] = {}
    for t, tn in zip(TIMESTEPS, NAME_TIMESTEPS):
        ckp_name = f"int_mlp_{traj}_t{tn}_x1167.pt"
        int_nfs[traj][t] = load_nf(f"{CKP_DIR}/{ckp_name}", device)

### Autoencoders

In [27]:
# AE
int_ae = f"{BASE_PATH}/autoencoders/ae/best.pth"
int_ae, _, int_ae_cfg = load_autoencoder(int_ae, device=device, load_peft=True)
int_ae.eval()

# VQ-VAE
int_vqvae = f"{BASE_PATH}/autoencoders/vqvae/best.pth"
int_vqvae, _, int_vqvae_cfg = load_autoencoder(int_vqvae, device=device, load_peft=True)
int_vqvae.eval()
pass

/system/apps/userenv/galletti/mhd/lib/python3.12/site-packages/peft/mapping_func.py:96: UserWarning: lora with eva initialization used with low_cpu_mem_usage=False. Setting low_cpu_mem_usage=True can improve the maximum batch size possible for eva initialization.
  warnings.warn(


Reconstructed PEFT model with method: eva
Loading model /system/user/publicwork/galletti/huggingface/autoencoders/ae/best.pth (stopped at epoch 20) 
Reconstructed PEFT model with method: eva
Loading model /system/user/publicwork/galletti/huggingface/autoencoders/vqvae/best.pth (stopped at epoch 100) 


In [6]:
from neugk.pinc.autoencoders.vapor import VAPOR, VQVAE, FNORefiner

vapor_ckp = f"{BASE_PATH}/autoencoders/vapor/best.pth"

num_hiddens = 256
num_residual_hiddens = 64
num_residual_layers = 4
embedding_dim = 1
num_embeddings = 512
commitment_cost = 0.25
decay = 0.99

# FNO Params
fno_modes1 = 8
fno_modes2 = 4
fno_width = 32

padding = [0, 0]

vqvae = VQVAE(
    in_channels=2,
    num_hiddens=num_hiddens,
    num_residual_layers=num_residual_layers,
    num_residual_hiddens=num_residual_hiddens,
    num_embeddings=num_embeddings,
    embedding_dim=embedding_dim,
    commitment_cost=commitment_cost,
    decay=decay,
    padding=padding,
).to(device)

fno_refiner = FNORefiner(
    in_channels=2,
    out_channels=2,
    modes1=fno_modes1,
    modes2=fno_modes2,
    width=fno_width,
    num_blocks=[2, 2, 2, 2],
).to(device)

vapor = VAPOR(vqvae, fno_refiner)
vapor.load_state_dict(torch.load(vapor_ckp))

<All keys matched successfully>

## Utils

In [7]:
def ml_eval(
    pred_df: torch.Tensor,
    gt_df: torch.Tensor,
    pred_phi: torch.Tensor,
    gt_phi: torch.Tensor,
    pred_eflux: torch.Tensor,
    gt_eflux: torch.Tensor,
    compressed_size: int = None,
):
    metrics = {}

    metrics["mse"] = ((pred_df.cpu() - gt_df.cpu()) ** 2).mean()
    metrics["l1"] = (pred_df.cpu() - gt_df.cpu()).abs().mean()
    metrics["psnr"] = 10 * torch.log10(gt_df.max() ** 2 / metrics["mse"])
    metrics["phi_mse"] = ((pred_phi - gt_phi) ** 2).mean()
    metrics["phi_l1"] = (pred_phi - gt_phi).abs().mean()
    metrics["phi_psnr"] = 10 * torch.log10(gt_phi.max() ** 2 / metrics["phi_mse"])
    metrics["eflux_l1"] = (pred_eflux - gt_eflux).abs().mean()

    if compressed_size is not None:
        num_pixels = pred_df.numel()
        metrics["bpp"] = (compressed_size * 8) / num_pixels

    return metrics

In [20]:
@torch.no_grad()
def run_eval_diagnostics(
    trajectories: str,
    timesteps: Sequence[int],
    model: Optional[Callable] = None,
    model_name: Optional[str] = None,
    train_norm_stats: Optional[Dict] = None,
):
    if model_name is None:
        model_name = "GT"
    ml_metrics = defaultdict(list)
    phy_diag = {}
    for traj in tqdm(trajectories):
        data = CycloneNFDataset(
            traj,
            timesteps=timesteps,
            path=f"{BASE_PATH}/h5",
            normalize=None,
            realpotens=True,
        )

        sgrid = np.loadtxt(f"/restricteddata/ukaea/gyrokinetics/raw/{traj}/sgrid")
        DS = sgrid[1] - sgrid[0]

        gt_dfs = (
            [data.df[:, t] for t in range(data.df.shape[1])]
            if data.ndim > 5
            else [data.df]
        )
        compressed_size = None
        if model_name == "GT":
            dfs = gt_dfs

        if "NF" in model_name:
            # nf reconstruct
            dfs = []
            compressed_size = 0
            for t in range(len(timesteps)):
                # TODO load config
                real_t = timesteps[t]
                if "ZFP" in model_name:
                    nf = model[traj][real_t].to(device)
                    nf, _, nf_zfp_size = compress_weights(nf, tolerance=1e-3)
                    compressed_size += nf_zfp_size
                if "ZipNN" in model_name:
                    nf = model[traj][real_t].to(device)
                    nf, _, nf_zipnn_size = compress_weights(nf, method="zipnn")
                    compressed_size += nf_zipnn_size
                else:
                    nf = model[traj][real_t].to(device)
                    compressed_size += sum(p.nbytes for p in nf.parameters())
                # create new normalized dataset
                nf_data = CycloneNFDataset(
                    traj,
                    path=f"{BASE_PATH}/h5",
                    timesteps=real_t,
                    normalize="zscore",
                    normalize_coords=False,
                    realpotens=True,
                )
                # automatically denormalizes
                dfs.append(sample_field(nf, nf_data, device).cpu())

                torch.cuda.empty_cache()

        if "AE" in model_name or "VQ-VAE" in model_name:
            dfs = []
            compressed_size = 0

            for t in range(data.df.shape[1]):
                ae, cfg = model
                ae = ae.to(device)
                if not train_norm_stats:
                    traindata_ae = CycloneAEDataset(
                        path=cfg.dataset.path,
                        split="train",
                        input_fields=["df", "phi", "flux"],
                        random_seed=cfg.seed,
                        normalization=cfg.dataset.normalization,
                        normalization_scope=cfg.dataset.normalization_scope,
                        trajectories=cfg.dataset.training_trajectories,
                        separate_zf=cfg.dataset.separate_zf,
                        real_potens=True,
                        cond_filters=vars(cfg.dataset.training_cond_filters),
                        stage=cfg.stage,
                        conditions=["itg", "dg", "s_hat", "q"],
                    )
                    train_norm_stats = traindata_ae.norm_stats

                valdata_ae = CycloneAEDataset(
                    path=f"{BASE_PATH}/h5",
                    split="train",
                    input_fields=["df", "phi", "flux"],
                    random_seed=cfg.seed,
                    normalization=cfg.dataset.normalization,
                    normalization_scope=cfg.dataset.normalization_scope,
                    normalization_stats=train_norm_stats,
                    trajectories=[f"{traj}.h5"],
                    separate_zf=cfg.dataset.separate_zf,
                    real_potens=True,
                    stage=cfg.stage,
                    conditions=["itg", "dg", "s_hat", "q"],
                )
                # ae reconstruct
                sample = valdata_ae[t]
                df = sample.df.unsqueeze(0).to(device)
                condition = sample.conditioning.unsqueeze(0).to(device)
                if "VAPOR" in model_name and valdata_ae.separate_zf:
                    df = df[:, [0, 1]] + df[:, [2, 3]]
                ae_df = ae(df, condition=condition)["df"].cpu().squeeze(0)
                # reconstruct separated zonal flow for vapor
                if "VAPOR" in model_name and valdata_ae.separate_zf:
                    ae_df = ae_df.numpy()
                    zf = np.repeat(
                        ae_df.mean(axis=-1, keepdims=True),
                        repeats=ae_df.shape[-1],
                        axis=-1,
                    )
                    ae_df = torch.from_numpy(np.concatenate([zf, ae_df - zf], axis=0))
                # important: denormalize
                ae_df = valdata_ae.denormalize(0, df=ae_df)
                if ae_df.shape[0] == 4:
                    ae_df = ae_df[[0, 1]] + ae_df[[2, 3]]
                dfs.append(ae_df)

                torch.cuda.empty_cache()

                if "VQ-VAE" in model_name:
                    model_output = ae(df, condition=condition)
                    indices = model_output["vq_indices"]

                    # (codebook_size=8192 fits in int16)
                    indices_int16 = indices.to(torch.int16)
                    compressed_size += indices_int16.nbytes
                if "VAPOR" in model_name:
                    # NOTE: hardcoded for now
                    compressed_size += data.full_df.nbytes / 64
                else:
                    # Regular AE
                    latent = ae.encode(df, condition=condition)[0]
                    compressed_size += latent.nbytes

        if model_name in ["ZFP", "Wavelet", "PCA", "JPEG2000"]:
            dfs = []
            compressed_size = 0
            for t in range(len(timesteps)):
                tdf = data.full_df[:, t] if data.ndim > 5 else data.full_df
                recon, _, cs = model(tdf)
                dfs.append(recon)
                compressed_size += cs
        if compressed_size:
            ml_metrics["cr"].append(data.full_df.nbytes / compressed_size)

        geom = {k: v.unsqueeze(0) for k, v in data.geom.items()}
        integral = FluxIntegral(
            real_potens=True, flux_fields=True, spectral_potens=True
        )

        # physics metrics
        phy_diag[traj] = defaultdict(list)
        phy_diag_gt = []
        for pred_df, gt_df in zip(dfs, gt_dfs):
            pred_phi, (_, pred_eflux, _) = integral(geom, pred_df.unsqueeze(0))
            pred_d = diagnostics(pred_phi.squeeze(), pred_eflux.squeeze(), ds=DS)
            pred_d = {k: v.numpy() for k, v in pred_d.items()}
            phy_diag[traj]["phi"].append(np.abs(pred_phi[:, 7, :].squeeze()))
            phy_diag[traj]["kxspec"].append(pred_d["kxspec"])
            phy_diag[traj]["kyspec"].append(pred_d["kyspec"])
            phy_diag[traj]["qspec"].append(pred_d["qspec"])

            if model_name != "GT":
                gt_phi, (_, gt_eflux, _) = integral(geom, gt_df.unsqueeze(0))
                gt_d = diagnostics(gt_phi.squeeze(), gt_eflux.squeeze(), ds=DS)
                gt_d = {k: v.numpy() for k, v in gt_d.items()}
                phy_diag_gt.append(gt_d)

        # time-averaged metrics
        if model_name != "GT":
            from scipy.stats import pearsonr, spearmanr, wasserstein_distance

            pred_kyspec_ta = np.stack(phy_diag[traj]["kyspec"], 0).mean(0)
            pred_qspec_ta = np.stack(phy_diag[traj]["qspec"], 0).mean(0)
            gt_kyspec_ta = np.stack([d["kyspec"] for d in phy_diag_gt], 0).mean(0)
            gt_qspec_ta = np.stack([d["qspec"] for d in phy_diag_gt], 0).mean(0)

            ml_metrics["kyspec_pc"].append(pearsonr(pred_kyspec_ta, gt_kyspec_ta)[0])
            ml_metrics["qspec_pc"].append(pearsonr(pred_qspec_ta, gt_qspec_ta)[0])

            ml_metrics["kyspec_l1"].append(np.abs(pred_kyspec_ta - gt_kyspec_ta).sum())
            ml_metrics["qspec_l1"].append(np.abs(pred_qspec_ta - gt_qspec_ta).sum())

            ml_metrics["kyspec_sc"].append(spearmanr(pred_kyspec_ta, gt_kyspec_ta)[0])
            ml_metrics["qspec_sc"].append(spearmanr(pred_qspec_ta, gt_qspec_ta)[0])

            pred_kyspec_ta /= pred_kyspec_ta.sum()
            gt_kyspec_ta /= gt_kyspec_ta.sum()
            pred_qspec_ta /= pred_qspec_ta.sum()
            gt_qspec_ta /= gt_qspec_ta.sum()
            ml_metrics["kyspec_wd"].append(
                wasserstein_distance(pred_kyspec_ta, gt_kyspec_ta)
            )
            ml_metrics["qspec_wd"].append(
                wasserstein_distance(pred_qspec_ta, gt_qspec_ta)
            )

        # ml metrics
        integral = FluxIntegral()
        if model_name != "GT":
            for pred_df, gt_df in zip(dfs, gt_dfs):
                pred_phi, (_, pred_eflux, _) = integral(geom, pred_df.unsqueeze(0))
                gt_phi, (_, gt_eflux, _) = integral(geom, gt_df.unsqueeze(0))

                ml_eval_dict = ml_eval(
                    pred_df,
                    gt_df,
                    pred_phi,
                    gt_phi,
                    pred_eflux,
                    gt_eflux,
                    compressed_size,
                )
                for k, v in ml_eval_dict.items():
                    ml_metrics[k].append(v)

            if len(gt_dfs) > 1:
                gt_df_ = np.stack([d.cpu().numpy() for d in gt_dfs])
                pred_df_ = np.stack([d.cpu().numpy() for d in dfs])
                ml_metrics["endpoint"].append(endpoint_error(gt_df_, pred_df_))

    ml_metrics = {k: (np.mean(v), np.std(v)) for k, v in ml_metrics.items()}
    if model_name != "GT":
        ml_metrics["bpp"] = ml_metrics["bpp"] * len(dfs)
    print(model_name, "done!")
    return {model_name: phy_diag}, {model_name: ml_metrics}

## Evaluation

In [9]:
full_diagnostics = pickle.load(open("vapor_diagnostics_nf.pkl", "rb"))
full_metrics = pickle.load(open("vapor_metrics_nf.pkl", "rb"))

In [31]:
full_diagnostics = {}
full_metrics = {}

gt_diag, _ = run_eval_diagnostics(TRAJECTORIES, timesteps=TIMESTEPS)
full_diagnostics.update(gt_diag)

100%|██████████| 20/20 [05:42<00:00, 17.12s/it]


GT done!


In [32]:
# ZFP
zfp_diag, zfp_metrics = run_eval_diagnostics(
    TRAJECTORIES, timesteps=TIMESTEPS, model=zfp_recon, model_name="ZFP"
)
full_diagnostics.update(zfp_diag)
full_metrics.update(zfp_metrics)

# wavelet
wft_diag, wft_metrics = run_eval_diagnostics(
    TRAJECTORIES, timesteps=TIMESTEPS, model=wavelet_recon, model_name="Wavelet"
)
full_diagnostics.update(wft_diag)
full_metrics.update(wft_metrics)

# PCA
pca_diag, pca_metrics = run_eval_diagnostics(
    TRAJECTORIES, timesteps=TIMESTEPS, model=pca_recon, model_name="PCA"
)
full_diagnostics.update(pca_diag)
full_metrics.update(pca_metrics)

# JPEG2000
jp2k_diag, jp2k_metrics = run_eval_diagnostics(
    TRAJECTORIES, timesteps=TIMESTEPS, model=jpeg2000_recon, model_name="JPEG2000"
)
full_diagnostics.update(jp2k_diag)
full_metrics.update(jp2k_metrics)

100%|██████████| 20/20 [10:56<00:00, 32.84s/it]


ZFP done!


100%|██████████| 20/20 [23:22<00:00, 70.10s/it]


Wavelet done!


100%|██████████| 20/20 [11:53<00:00, 35.67s/it]


PCA done!


100%|██████████| 20/20 [41:50<00:00, 125.53s/it]


JPEG2000 done!


In [33]:
pickle.dump(full_diagnostics, open("trad_diagnostics_nf.pkl", "wb"))
pickle.dump(full_metrics, open("trad_metrics_nf.pkl", "wb"))

In [28]:
# NF
nf_diag, nf_metrics = run_eval_diagnostics(
    TRAJECTORIES, timesteps=TIMESTEPS, model=nfs, model_name="NF"
)
full_diagnostics.update(nf_diag)
full_metrics.update(nf_metrics)

# NF + int
nf_diag, nf_metrics = run_eval_diagnostics(
    TRAJECTORIES, timesteps=TIMESTEPS, model=int_nfs, model_name="PINC-NF"
)
full_diagnostics.update(nf_diag)
full_metrics.update(nf_metrics)

100%|██████████| 20/20 [19:11<00:00, 57.55s/it]


NF done!


100%|██████████| 20/20 [16:32<00:00, 49.62s/it]


PINC-NF done!


In [29]:
pickle.dump(full_diagnostics, open("nf_diagnostics_nf.pkl", "wb"))
pickle.dump(full_metrics, open("nf_metrics_nf.pkl", "wb"))

In [28]:
# AE + INT
int_ae_diag, int_ae_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    model=(int_ae, int_ae_cfg),
    model_name="AE + EVA",
    train_norm_stats=pickle.load(open(f"{BASE_PATH}/hf_norm_stats.pkl", "rb")),
)
full_diagnostics.update(int_ae_diag)
full_metrics.update(int_ae_metrics)

# VQ-VAE + INT
int_vqvae_diag, int_vqvae_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    model=(int_vqvae, int_vqvae_cfg),
    model_name="VQ-VAE + EVA",
    train_norm_stats=pickle.load(open(f"{BASE_PATH}/hf_norm_stats.pkl", "rb")),
)
full_diagnostics.update(int_vqvae_diag)
full_metrics.update(int_vqvae_metrics)

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [16:08<00:00, 48.45s/it]


VQ-VAE + EVA done!


In [12]:
# VAPOR
int_vqvae_cfg.dataset.separate_zf = False
vapor_diag, vapor_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    model=(vapor, int_ae_cfg),
    model_name="AE VAPOR",
    train_norm_stats=pickle.load(open(f"{BASE_PATH}/hf_norm_stats.pkl", "rb")),
)
full_diagnostics.update(vapor_diag)
full_metrics.update(vapor_metrics)

  0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_1835066/3416469960.py:105: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  ae_df = np.array(ae_df)
100%|██████████| 20/20 [52:27<00:00, 157.38s/it] 


AE VAPOR done!


In [26]:
pickle.dump(full_diagnostics, open("vapor_diagnostics_nf.pkl", "wb"))
pickle.dump(full_metrics, open("vapor_metrics_nf.pkl", "wb"))

## Table

In [30]:
ADD_STD = True

metrics_keys = ["psnr", "endpoint", "eflux_l1", "phi_psnr", "kyspec_wd", "qspec_wd"]

direction = {
    "l1": "min",
    "psnr": "max",
    "bpp": "min",
    "endpoint": "min",
    "eflux_l1": "min",
    "phi_psnr": "max",
    "kyspec_pc": "max",
    "qspec_pc": "max",
    "kyspec_sc": "max",
    "qspec_sc": "max",
    "kyspec_wd": "min",
    "qspec_wd": "min",
    "kyspec_shd": "min",
    "qspec_shd": "min",
    "kyspec_l1": "min",
    "qspec_l1": "min",
}

values_per_metric = {k: [] for k in metrics_keys}
for m, vals in full_metrics.items():
    if m != "GT":
        for k in metrics_keys:
            values_per_metric[k].append(vals[k][0])

# Compute column medians for thresholding
medians = {k: np.median(values_per_metric[k]) for k in metrics_keys}

best_idx, second_idx = {}, {}
for k in metrics_keys:
    arr = np.array(values_per_metric[k])
    if direction[k] == "min":
        order = np.argsort(arr)
    else:
        order = np.argsort(-arr)
    best_idx[k] = order[0]
    second_idx[k] = order[1]

for i, m in enumerate(full_metrics):
    vals = full_metrics[m]
    cr = f"{int(vals['cr'][0])}$\\times$"

    row_entries = []
    for j, k in enumerate(metrics_keys):
        val_mean, val_std = vals[k][0], vals[k][1]
        if k == "bpp":
            formatted = f"{val_mean / 34:.4f}"
        elif "spec" in k or "endpoint" in k:
            formatted = f"{val_mean:.3f}"
        else:
            formatted = f"{val_mean:.2f}"
        if ADD_STD and val_std > 1e-3 and k != "bpp":
            formatted += r"$_{\pm " + f"{val_std:.2f}" + r"}$"

        if i == best_idx[k]:
            formatted = r"\textbf{" + formatted + "}"
        elif i == second_idx[k]:
            formatted = r"\underline{" + formatted + "}"

        row_entries.append(formatted)

    # print(rf"| {m:<8} | {cr} | " + " | ".join(row_entries) + "  | ")
    print(rf"{m:<8} & {cr} & " + " & ".join(row_entries) + rf" \\")

ZFP      & 901$\times$ & 28.97$_{\pm 1.09}$ & 0.169$_{\pm 0.10}$ & 107.48$_{\pm 49.35}$ & -16.20$_{\pm 7.09}$ & 0.020$_{\pm 0.01}$ & 0.116$_{\pm 0.20}$ \\
Wavelet  & 936$\times$ & 33.07$_{\pm 1.18}$ & 0.063$_{\pm 0.04}$ & 107.74$_{\pm 49.51}$ & -13.24$_{\pm 9.20}$ & 0.020$_{\pm 0.01}$ & \textbf{0.010$_{\pm 0.00}$} \\
PCA      & 1020$\times$ & 32.09$_{\pm 0.98}$ & 0.091$_{\pm 0.06}$ & 67.60$_{\pm 36.08}$ & -10.22$_{\pm 6.89}$ & 0.020$_{\pm 0.01}$ & 0.011$_{\pm 0.00}$ \\
JPEG2000 & 1000$\times$ & 34.33$_{\pm 0.95}$ & 0.062$_{\pm 0.05}$ & 103.91$_{\pm 44.12}$ & -20.85$_{\pm 6.50}$ & 0.020$_{\pm 0.01}$ & 0.035$_{\pm 0.03}$ \\
NF       & 1167$\times$ & \textbf{36.91$_{\pm 0.93}$} & \textbf{0.039$_{\pm 0.03}$} & 61.50$_{\pm 16.91}$ & 1.24$_{\pm 5.99}$ & 0.017$_{\pm 0.01}$ & 0.017$_{\pm 0.00}$ \\
PINC-NF  & 1167$\times$ & \underline{35.76$_{\pm 1.38}$} & \underline{0.046$_{\pm 0.03}$} & \textbf{2.18$_{\pm 8.33}$} & \textbf{13.50$_{\pm 4.44}$} & \textbf{0.006$_{\pm 0.00}$} & 0.015$_{\pm 0.00}$

In [31]:
metrics_keys = ["l1", "bpp", "phi_l1", "kyspec_pc", "qspec_pc", "kyspec_l1", "qspec_l1"]

direction = {
    "l1": "min",
    "psnr": "max",
    "bpp": "min",
    "phi_l1": "min",
    "eflux_l1": "min",
    "phi_psnr": "max",
    "kyspec_pc": "max",
    "qspec_pc": "max",
    "kyspec_sc": "max",
    "qspec_sc": "max",
    "kyspec_wd": "min",
    "qspec_wd": "min",
    "kyspec_shd": "min",
    "qspec_shd": "min",
    "kyspec_l1": "min",
    "qspec_l1": "min",
}

values_per_metric = {k: [] for k in metrics_keys}
for m, vals in full_metrics.items():
    for k in metrics_keys:
        values_per_metric[k].append(vals[k])

# best_idx, second_idx = {}, {}
# for k in metrics_keys:
#     arr = np.array(values_per_metric[k])
#     if direction[k] == "min":
#         order = np.argsort(arr)
#     else:
#         order = np.argsort(-arr)
#     best_idx[k] = order[0]
#     second_idx[k] = order[1]

# print(r"\begin{tabular}{l|c|cccc}")
# print(r"\toprule")
# print(
#     r"         & \multicolumn{1}{c}{Integrals $\boldsymbol{\phi}$}                                              & \multicolumn{4}{c}{Turbulence $Q^{\text{spec}}, k_y^{\text{spec}}$}                                        \\ \midrule"
# )
# print(
#     r"         & $\text{L1}(\boldsymbol{\phi})$ $\downarrow$ & $\text{PC}(\overline{k_y^{\text{spec}}})$ $\uparrow$ & $\text{PC}(\overline{Q^{\text{spec}}})$ $\uparrow$ & $\text{L1}(\overline{k_y^{\text{spec}}})$ $\uparrow$ & $\text{L1}(\overline{Q^{\text{spec}}})$ $\uparrow$ \\ \midrule"
# )

for i, m in enumerate(full_metrics):
    vals = full_metrics[m]
    cr = f"{int(vals['cr'][0])}$\\times$"

    row_entries = []
    for j, k in enumerate(metrics_keys):
        val_mean, val_std = vals[k][0], vals[k][1]
        if k == "bpp":
            formatted = f"{val_mean:.3f}"
        elif "spec" in k:
            formatted = f"{val_mean:.4f}"
        else:
            formatted = f"{val_mean:.2f}"
        if val_std > 1e-3 and k != "bpp":
            formatted += r"$_{\pm " + f"{val_std:.2f}" + r"}$"

        # if i == best_idx[k]:
        #     formatted = r"\textbf{" + formatted + "}"
        # elif i == second_idx[k]:
        #     formatted = r"\underline{" + formatted + "}"

        row_entries.append(formatted)

    rule = ""
    if "VQ-VAE" in m:
        rule = r"\midrule"
    if i == len(full_metrics) - 1:
        rule = r"\bottomrule"

    print(rf"{m:<8} & " + " & ".join(row_entries) + rf" \\{rule}")

# print(r"\end{tabular}")

ZFP      & 0.72$_{\pm 0.19}$ & 0.798 & 1112.85$_{\pm 1058.86}$ & 0.9287$_{\pm 0.10}$ & -0.0053$_{\pm 0.73}$ & 436992.2500$_{\pm 859162.75}$ & 107.5087$_{\pm 40.90}$ \\
Wavelet  & 0.50$_{\pm 0.12}$ & 0.890 & 749.26$_{\pm 854.38}$ & 0.9291$_{\pm 0.10}$ & -0.9628$_{\pm 0.05}$ & 390304.1875$_{\pm 826847.88}$ & 107.7359$_{\pm 40.99}$ \\
PCA      & 0.52$_{\pm 0.14}$ & 0.658 & 446.01$_{\pm 418.89}$ & 0.9290$_{\pm 0.10}$ & 0.8705$_{\pm 0.38}$ & 99478.8125$_{\pm 156164.62}$ & 67.4491$_{\pm 24.02}$ \\
JPEG2000 & 0.51$_{\pm 0.13}$ & 0.672 & 1782.05$_{\pm 1523.60}$ & 0.9280$_{\pm 0.10}$ & -0.0363$_{\pm 0.82}$ & 863066.6875$_{\pm 1273776.88}$ & 103.9146$_{\pm 36.75}$ \\
NF       & 0.33$_{\pm 0.09}$ & 0.576 & 136.67$_{\pm 125.50}$ & 0.9392$_{\pm 0.08}$ & 0.9735$_{\pm 0.02}$ & 6072.0742$_{\pm 10074.11}$ & 61.4989$_{\pm 14.96}$ \\
PINC-NF  & 0.37$_{\pm 0.08}$ & 0.576 & 26.52$_{\pm 24.35}$ & 0.9870$_{\pm 0.02}$ & 0.9797$_{\pm 0.02}$ & 237.3242$_{\pm 391.57}$ & 49.4779$_{\pm 15.88}$ \\
AE + EVA & 0.45$_

## Temporal table

In [ ]:
metrics_keys = ["endpoint", "dmd_bmode", "dmd_dmd"]

direction = {"endpoint": "min", "dmd_bmode": "min", "dmd_dmd": "min"}

values_per_metric = {k: [] for k in metrics_keys}
for m, vals in full_metrics.items():
    for k in metrics_keys:
        values_per_metric[k].append(vals[k])

medians = {k: np.median(values_per_metric[k]) for k in metrics_keys}

for i, (m, vals) in enumerate(full_metrics.items()):
    row_entries = []
    for j, k in enumerate(metrics_keys):
        val_mean, val_std = vals[k][0], vals[k][1]
        formatted = f"{val_mean:.3f}" + r"$_{\pm " + f"{val_std:.2f}" + r"}$"
        row_entries.append(formatted)

    rule = ""
    if "VQ-VAE" in m:
        rule = r" \midrule"
    if i == len(full_metrics) - 1:
        rule = r" \bottomrule"

    print(f"    {m:<16} & " + " & ".join(row_entries) + rf" \\{rule}")

In [ ]:
raise NotImplementedError

## Quantitative plot

In [ ]:
TIME = np.loadtxt("/restricteddata/ukaea/gyrokinetics/raw/iteration_13/time.dat")

In [ ]:
MODEL_WHITELIST = ["NF", "VQ-VAE + EVA", "ZFP", "PCA"]
TRAJ_WHITELIST = [
    #   "iteration_13",
    #   "iteration_115",
    #   "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    #   "iteration_200",
    #   "iteration_210",
    #   "iteration_212",
]
fig, ax = plt.subplots(
    len(MODEL_WHITELIST),
    len(TRAJ_WHITELIST),
    figsize=(5 * len(TRAJ_WHITELIST), int(1.7 * len(MODEL_WHITELIST))),
    gridspec_kw={"hspace": 0.0},
)
plasma = matplotlib.colormaps["plasma"]

row = 0
for model_name in MODEL_WHITELIST:
    cyc_diag = full_diagnostics[model_name]
    if "EVA" in model_name:
        model_name = model_name.replace(" + EVA", "")

    col = 0
    for case_name, diag in cyc_diag.items():
        if case_name not in TRAJ_WHITELIST:
            continue
        kys = diag["kyspec"]
        gt_kys = full_diagnostics["GT"][case_name]["kyspec"]

        n_lines = len(kys)
        for t in range(n_lines):
            color = plasma((t + 0.1) / max(1, n_lines - 0.5))
            ax[row, col].plot(
                gt_kys[t][1:],
                linewidth=2,
                color=color,
                alpha=0.5,
                zorder=0,
                linestyle="--",
            )
            model_curve = kys[t][1:]
            gt_curve = gt_kys[t][1:]
            ax[row, col].plot(model_curve, linewidth=3, color=color, zorder=1)

            # corr, _ = pearsonr(model_curve, gt_curve)
            # corr, _ = pearsonr(np.log(model_curve), np.log(gt_curve))
            # ax[row, col].text(
            #     0.98,
            #     0.98 - t * 0.1,
            #     rf"$PC={corr:.2f}$",
            #     color=color,
            #     fontsize=14,
            #     transform=ax[row, col].transAxes,
            #     ha="right",
            #     va="top",
            # )

        ax[row, col].tick_params(axis="x", which="both", labelsize=12)
        ax[row, col].tick_params(axis="y", which="both", labelsize=12)

        if col == 0 and row == (len(MODEL_WHITELIST) // 2):
            ax[row, col].set_ylabel(r"$|\phi(k_y)|^2$", fontsize=26)

        if row == len(MODEL_WHITELIST) - 1:
            ax[row, col].set_xlabel(r"$k_y$", fontsize=26)
        else:
            ax[row, col].set_xticklabels([])
            ax[row, col].tick_params(axis="x", which="both", length=0)

        ymin, ymax = ax[row, col].get_ylim()
        yticks = [
            ymin + 0.1 * (ymax - ymin),
            ymin + 0.5 * (ymax - ymin),
            ymin + 0.9 * (ymax - ymin),
        ]
        ax[row, col].set_yticks([round(yt, 1) for yt in yticks])
        ax[row, col].grid(True, linestyle="--", alpha=0.6)

        ax[row, col].set_xscale("log")
        ax[row, col].set_yscale("log")

        # ax[row, col].set_xlim(0, 15)

        col += 1
    col = col - 1

    ax[row, -1].text(
        1.05,
        0.5,
        model_name,
        transform=ax[row, -1].transAxes,
        fontsize=22,
        rotation=270,
        va="center",
        ha="left",
    )
    if row != len(MODEL_WHITELIST) - 1:
        ax[row, col].set_xticks([])

    row += 1


t_indices = [TIME[t] for t in TIMESTEPS]
n_lines = max(
    len(kys)
    for cyc in full_diagnostics.values()
    for _, diag in cyc.items()
    for kys in [diag["kyspec"]]
)

handles = []
labels = []
for t, physical_time in enumerate(t_indices):
    color = plasma((t + 0.1) / max(1, n_lines - 0.5))
    handles.append(matplotlib.lines.Line2D([0], [0], color=color, lw=3))
    labels.append(rf"$t={physical_time:.1f}" + r"R/V_{\mathrm{r}}$")

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    fontsize=26,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
)
fig.savefig("cascade_iclr.pdf", bbox_inches="tight")

In [ ]:
MODEL_WHITELIST = ["NF", "VQ-VAE + EVA", "ZFP", "Wavelet"]
TRAJ_WHITELIST = [
    #   "iteration_13",
    #   "iteration_115",
    #   "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    #   "iteration_200",
    #   "iteration_210",
    #   "iteration_212",
]
fig, ax = plt.subplots(
    len(MODEL_WHITELIST),
    len(TRAJ_WHITELIST),
    figsize=(5 * len(TRAJ_WHITELIST), int(1.7 * len(MODEL_WHITELIST))),
    gridspec_kw={"hspace": 0.0},
)
plasma = matplotlib.colormaps["plasma"]

row = 0

for model_name in MODEL_WHITELIST:
    cyc_diag = full_diagnostics[model_name]
    if "EVA" in model_name:
        model_name = model_name.replace(" + EVA", "")

    col = 0
    for case_name, diag in cyc_diag.items():
        if case_name not in TRAJ_WHITELIST:
            continue
        kys = diag["qspec"]
        gt_kys = full_diagnostics["GT"][case_name]["qspec"]

        n_lines = len(kys)
        for t in range(n_lines):
            color = plasma((t + 0.1) / max(1, n_lines - 0.5))
            ax[row, col].plot(
                gt_kys[t][1:],
                linewidth=2,
                color=color,
                alpha=0.5,
                zorder=0,
                linestyle="--",
            )
            model_curve = kys[t][1:]
            gt_curve = gt_kys[t][1:]
            model_curve = np.clip(
                model_curve, model_curve[model_curve > 0].min(), np.inf
            )
            ax[row, col].plot(model_curve, linewidth=3, color=color, zorder=1)

            # corr, _ = pearsonr(model_curve, gt_curve)
            # corr, _ = pearsonr(np.log(model_curve), np.log(gt_curve))
            # ax[row, col].text(
            #     0.98,
            #     0.98 - t * 0.1,
            #     rf"$PC={corr:.2f}$",
            #     color=color,
            #     fontsize=14,
            #     transform=ax[row, col].transAxes,
            #     ha="right",
            #     va="top",
            # )

        ax[row, col].tick_params(axis="x", which="both", labelsize=12)
        ax[row, col].tick_params(axis="y", which="both", labelsize=12)

        if col == 0 and row == (len(MODEL_WHITELIST) // 2):
            ax[row, col].set_ylabel(r"$Q(k_y)$", fontsize=26)

        if row == len(MODEL_WHITELIST) - 1:
            ax[row, col].set_xlabel(r"$k_y$", fontsize=26)
        else:
            ax[row, col].set_xticklabels([])
            ax[row, col].tick_params(axis="x", which="both", length=0)

        ymin, ymax = ax[row, col].get_ylim()
        yticks = [
            ymin + 0.1 * (ymax - ymin),
            ymin + 0.5 * (ymax - ymin),
            ymin + 0.9 * (ymax - ymin),
        ]
        ax[row, col].set_yticks([round(yt, 1) for yt in yticks])
        ax[row, col].grid(True, linestyle="--", alpha=0.6)

        ax[row, col].set_xscale("log")
        ax[row, col].set_yscale("log")

        # ax[row, col].set_xlim(0, 15)

        col += 1
    col = col - 1

    ax[row, -1].text(
        1.05,
        0.5,
        model_name,
        transform=ax[row, -1].transAxes,
        fontsize=22,
        rotation=270,
        va="center",
        ha="left",
    )
    if row != len(MODEL_WHITELIST) - 1:
        ax[row, col].set_xticks([])

    row += 1


t_indices = [TIME[t] for t in TIMESTEPS]
n_lines = max(
    len(kys)
    for cyc in full_diagnostics.values()
    for _, diag in cyc.items()
    for kys in [diag["qspec"]]
)

handles = []
labels = []
for t, physical_time in enumerate(t_indices):
    color = plasma((t + 0.1) / max(1, n_lines - 0.5))
    handles.append(matplotlib.lines.Line2D([0], [0], color=color, lw=3))
    labels.append(rf"$t={physical_time:.1f}" + r"R/V_{\mathrm{r}}$")

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    fontsize=26,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
)
fig.savefig("qcascade_iclr.pdf", bbox_inches="tight")

## Extra plot

In [ ]:
MODEL_WHITELIST = [
    "GT",
    "NF",
    "AE + EVA",
    "VQ-VAE + EVA",
    "ZFP",
    "Wavelet",
    "PCA",
    "JPEG2000",
]
TRAJ_WHITELIST = [
    "iteration_13",
    "iteration_115",
    "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    "iteration_200",
    "iteration_210",
    "iteration_212",
]
fig, ax = plt.subplots(
    len(MODEL_WHITELIST),
    len(TRAJ_WHITELIST),
    figsize=(5 * len(TRAJ_WHITELIST), int(2.5 * len(MODEL_WHITELIST))),
    gridspec_kw={"hspace": 0.0},
)
plasma = matplotlib.colormaps["plasma"]

row = 0

for model_name in MODEL_WHITELIST:
    cyc_diag = full_diagnostics[model_name]
    if "EVA" in model_name:
        model_name = model_name.replace(" + EVA", "")

    col = 0
    for case_name, diag in cyc_diag.items():
        if case_name not in TRAJ_WHITELIST:
            continue
        kys = diag["kyspec"]
        gt_kys = full_diagnostics["GT"][case_name]["kyspec"]

        n_lines = len(kys)

        for t in range(n_lines):
            color = plasma((t + 0.1) / max(1, n_lines - 0.5))
            model_curve = kys[t][1:]
            gt_curve = gt_kys[t][1:]
            if model_curve.sum() > 0:
                model_curve = np.clip(
                    model_curve, model_curve[model_curve > 0].min(), np.inf
                )
            ax[row, col].plot(model_curve, linewidth=3, color=color, zorder=1)

            if model_name != "GT":
                ax[row, col].plot(
                    gt_kys[t][1:],
                    linewidth=2,
                    color=color,
                    alpha=0.5,
                    zorder=0,
                    linestyle="--",
                )
                corr, _ = pearsonr(model_curve, gt_curve)
                # corr, _ = pearsonr(np.log(model_curve), np.log(gt_curve))
                ax[row, col].text(
                    0.98,
                    0.98 - t * 0.08,
                    rf"$PC={corr:.2f}$",
                    color=color,
                    fontsize=14,
                    transform=ax[row, col].transAxes,
                    ha="right",
                    va="top",
                )

        if col == 0:
            ax[row, col].set_ylabel(r"$|\phi(k_y)|^2$", fontsize=24)

        if row == len(MODEL_WHITELIST) - 1:
            ax[row, col].set_xlabel(r"$k_y$", fontsize=24)
        else:
            ax[row, col].set_xticklabels([])
            ax[row, col].tick_params(axis="x", which="both", length=0)

        ymin, ymax = ax[row, col].get_ylim()
        yticks = [
            ymin + 0.1 * (ymax - ymin),
            ymin + 0.5 * (ymax - ymin),
            ymin + 0.9 * (ymax - ymin),
        ]
        ax[row, col].set_yticks([round(yt, 1) for yt in yticks])
        ax[row, col].grid(True, linestyle="--", alpha=0.6)

        ax[row, col].set_xscale("log")
        ax[row, col].set_yscale("log")
        col += 1
    col = col - 1

    ax[row, -1].text(
        1.05,
        0.5,
        model_name,
        transform=ax[row, -1].transAxes,
        fontsize=20,
        rotation=270,
        va="center",
        ha="left",
    )
    if row != len(MODEL_WHITELIST) - 1:
        ax[row, col].set_xticks([])

    row += 1


t_indices = [TIME[t] for t in TIMESTEPS]
n_lines = max(
    len(kys)
    for cyc in full_diagnostics.values()
    for _, diag in cyc.items()
    for kys in [diag["kyspec"]]
)

handles = []
labels = []
for t, physical_time in enumerate(t_indices):
    color = plasma((t + 0.1) / max(1, n_lines - 0.5))
    handles.append(matplotlib.lines.Line2D([0], [0], color=color, lw=3))
    labels.append(rf"$t={physical_time:.1f}" + r"R/V_{\mathrm{r}}$")

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    fontsize=24,
    frameon=False,
    bbox_to_anchor=(0.5, 0.92),
    # bbox_to_anchor=(0.5, 0.98),
)
fig.savefig("extra_cascade_iclr.pdf", bbox_inches="tight")

In [ ]:
MODEL_WHITELIST = [
    "GT",
    "NF",
    "AE + EVA",
    "VQ-VAE + EVA",
    "ZFP",
    "Wavelet",
    "PCA",
    "JPEG2000",
]
TRAJ_WHITELIST = [
    "iteration_13",
    "iteration_115",
    "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    "iteration_200",
    "iteration_210",
    "iteration_212",
]
fig, ax = plt.subplots(
    len(MODEL_WHITELIST),
    len(TRAJ_WHITELIST),
    figsize=(5 * len(TRAJ_WHITELIST), int(2.5 * len(MODEL_WHITELIST))),
    gridspec_kw={"hspace": 0.0},
)
plasma = matplotlib.colormaps["plasma"]

row = 0

for model_name in MODEL_WHITELIST:
    cyc_diag = full_diagnostics[model_name]
    if "EVA" in model_name:
        model_name = model_name.replace(" + EVA", "")

    col = 0
    for case_name, diag in cyc_diag.items():
        if case_name not in TRAJ_WHITELIST:
            continue
        kys = diag["qspec"]
        gt_kys = full_diagnostics["GT"][case_name]["qspec"]

        n_lines = len(kys)

        for t in range(n_lines):
            color = plasma((t + 0.1) / max(1, n_lines - 0.5))
            model_curve = kys[t][1:]
            gt_curve = gt_kys[t][1:]
            if model_curve.sum() > 0:
                model_curve = np.clip(
                    model_curve, model_curve[model_curve > 0].min(), np.inf
                )
            ax[row, col].plot(model_curve, linewidth=3, color=color, zorder=1)

            if model_name != "GT":
                ax[row, col].plot(
                    gt_kys[t][1:],
                    linewidth=2,
                    color=color,
                    alpha=0.5,
                    zorder=0,
                    linestyle="--",
                )
                corr, _ = pearsonr(model_curve, gt_curve)
                # corr, _ = pearsonr(np.log(model_curve), np.log(gt_curve))
                ax[row, col].text(
                    0.98,
                    0.98 - t * 0.08,
                    rf"$PC={corr:.2f}$",
                    color=color,
                    fontsize=14,
                    transform=ax[row, col].transAxes,
                    ha="right",
                    va="top",
                )

        if col == 0:
            ax[row, col].set_ylabel(r"$Q(k_y)$", fontsize=24)

        if row == len(MODEL_WHITELIST) - 1:
            ax[row, col].set_xlabel(r"$k_y$", fontsize=24)
        else:
            ax[row, col].set_xticklabels([])
            ax[row, col].tick_params(axis="x", which="both", length=0)

        ymin, ymax = ax[row, col].get_ylim()
        yticks = [
            ymin + 0.1 * (ymax - ymin),
            ymin + 0.5 * (ymax - ymin),
            ymin + 0.9 * (ymax - ymin),
        ]
        ax[row, col].set_yticks([round(yt, 1) for yt in yticks])
        ax[row, col].grid(True, linestyle="--", alpha=0.6)

        ax[row, col].set_xscale("log")
        ax[row, col].set_yscale("log")
        col += 1
    col = col - 1

    ax[row, -1].text(
        1.05,
        0.5,
        model_name,
        transform=ax[row, -1].transAxes,
        fontsize=20,
        rotation=270,
        va="center",
        ha="left",
    )
    if row != len(MODEL_WHITELIST) - 1:
        ax[row, col].set_xticks([])

    row += 1


t_indices = [TIME[t] for t in TIMESTEPS]
n_lines = max(
    len(kys)
    for cyc in full_diagnostics.values()
    for _, diag in cyc.items()
    for kys in [diag["kyspec"]]
)

handles = []
labels = []
for t, physical_time in enumerate(t_indices):
    color = plasma((t + 0.1) / max(1, n_lines - 0.5))
    handles.append(matplotlib.lines.Line2D([0], [0], color=color, lw=3))
    labels.append(rf"$t={physical_time:.1f}" + r"R/V_{\mathrm{r}}$")

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    fontsize=24,
    frameon=False,
    bbox_to_anchor=(0.5, 0.92),
    # bbox_to_anchor=(0.5, 0.98),
)
fig.savefig("extra_qcascade_iclr.pdf", bbox_inches="tight")